In [ ]:
# ==============================================================================
# PCA + t-SNE Visualization Before and After COMBAT
#STAGE C
# ==============================================================================

import sys
import subprocess
import importlib
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "openpyxl": "openpyxl",
    "sklearn": "scikit-learn"
}

for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ==============================================================================
# Paths
# ==============================================================================
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

FILE_BEFORE = BASE_DIR / "merged_common_genes.xlsx"
FILE_AFTER  = BASE_DIR / "combat_corrected_by_gse.xlsx"

# ==============================================================================
# Style
# ==============================================================================
sns.set_theme(style="white")
plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.linewidth": 0.8
})

# ==============================================================================
# Utilities
# ==============================================================================
def extract_gse(name):
    match = re.search(r"(GSE\d+)", str(name))
    return match.group(1) if match else "Other"


def load_expression_matrix(file_path):
    return pd.read_excel(file_path, index_col=0, engine="openpyxl").select_dtypes(include="number")


def plot_panel(ax, emb, labels, unique_gse, colors, zoom_factor):
    x_min, x_max = emb[:, 0].min(), emb[:, 0].max()
    y_min, y_max = emb[:, 1].min(), emb[:, 1].max()

    pad_x = zoom_factor * (x_max - x_min)
    pad_y = zoom_factor * (y_max - y_min)

    for gse in unique_gse:
        idx = np.where(labels == gse)[0]
        ax.scatter(
            emb[idx, 0],
            emb[idx, 1],
            s=28,
            color=colors[gse],
            alpha=0.9,
            linewidth=0,
            rasterized=True
        )

    ax.set_xlim(x_min - pad_x, x_max + pad_x)
    ax.set_ylim(y_min - pad_y, y_max + pad_y)
    ax.set_xticks([])
    ax.set_yticks([])


# ==============================================================================
# Main
# ==============================================================================
expr_before = load_expression_matrix(FILE_BEFORE)
expr_after  = load_expression_matrix(FILE_AFTER)

# Align gene order
expr_after = expr_after.loc[expr_before.index]

labels_before = np.array([extract_gse(c) for c in expr_before.columns])
labels_after  = np.array([extract_gse(c) for c in expr_after.columns])

X_before = expr_before.T.values
X_after  = expr_after.T.values

# Shared PCA space
pca = PCA(n_components=2, random_state=42)
pca_b = pca.fit_transform(X_before)
pca_a = pca.transform(X_after)

# Shared t-SNE space
X_all = np.vstack([X_before, X_after])
tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=42
)
tsne_all = tsne.fit_transform(X_all)
tsne_b = tsne_all[:len(X_before)]
tsne_a = tsne_all[len(X_before):]

unique_gse = sorted(set(labels_before))
palette = [
    "#1f77b4", "#d62728", "#2ca02c", "#ff7f0e",
    "#9467bd", "#17becf", "#e377c2", "#8c564b"
]
colors = {gse: palette[i % len(palette)] for i, gse in enumerate(unique_gse)}

fig, axes = plt.subplots(2, 2, figsize=(10, 6), dpi=300)

# Top row: Before COMBAT
plot_panel(axes[0, 0], pca_b,  labels_before, unique_gse, colors, zoom_factor=0.06)
plot_panel(axes[0, 1], tsne_b, labels_before, unique_gse, colors, zoom_factor=0.06)

# Bottom row: After COMBAT
plot_panel(axes[1, 0], pca_a,  labels_after,  unique_gse, colors, zoom_factor=0.02)
plot_panel(axes[1, 1], tsne_a, labels_after,  unique_gse, colors, zoom_factor=0.02)

# Column titles
axes[0, 0].set_title("PCA", fontsize=14, fontweight="bold")
axes[0, 1].set_title("t-SNE", fontsize=14, fontweight="bold")

# Row labels
axes[0, 0].text(
    -0.12, 0.5, "Before COMBAT",
    transform=axes[0, 0].transAxes,
    rotation=90, va="center", ha="right",
    fontsize=12, fontweight="bold"
)

axes[1, 0].text(
    -0.12, 0.5, "After COMBAT",
    transform=axes[1, 0].transAxes,
    rotation=90, va="center", ha="right",
    fontsize=12, fontweight="bold"
)

# Bottom legend
handles = [
    plt.Line2D([0], [0], marker="o", linestyle="", markersize=6, color=colors[gse])
    for gse in unique_gse
]

fig.legend(
    handles,
    unique_gse,
    loc="lower center",
    ncol=len(unique_gse),
    frameon=False,
    title="Cohort (GSE)",
    bbox_to_anchor=(0.5, 0.02)
)

# Tight layout similar to target style
plt.subplots_adjust(
    left=0.07,
    right=0.98,
    top=0.98,
    bottom=0.14,
    wspace=0.00,
    hspace=0.00
)

png_path = OUTPUT_DIR / "fig_pca_tsne_before_after_combat.png"
pdf_path = OUTPUT_DIR / "fig_pca_tsne_before_after_combat.pdf"

fig.savefig(png_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")

plt.show()

print(f"Figure saved: {png_path.name}")
print(f"Figure saved: {pdf_path.name}")

In [ ]:
# ==============================================================================
# RA Source-Domain Classification: Kernel-MLP Analysis Pipeline
# Pathway Selection Frequency & Multi-L1 Regularization Sensitivity
#STAGE D
# ==============================================================================

import sys
import subprocess
import warnings
import os
import random
import time
import json
from pathlib import Path

# --- Environment Setup & Warning Suppression ---
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

def setup_environment():
    """Ensures necessary packages are installed before execution."""
    try:
        import numpy as np
        if not np.__version__.startswith("1.26"): raise ImportError
    except ImportError:
        print("Configuring environment... Please wait.")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "numpy==1.26.4"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", 
                               "tensorflow", "gseapy", "pandas", "scikit-learn", "openpyxl", "matplotlib", "seaborn"])

setup_environment()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.constraints import NonNeg
from tensorflow.keras.regularizers import l1
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed
from gseapy import get_library

# ------------------------------------------------------------------------------
# Global Configuration
# ------------------------------------------------------------------------------
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUTPUT_ROOT = BASE_DIR / "outputs"
OUTPUT_ROOT.mkdir(exist_ok=True)

# File names (Ensure these files are in the same folder as the script)
EXPR_FILE = BASE_DIR / "combat_corrected_by_gse.xlsx"
PHENO_FILE = BASE_DIR / "pheno_raw.xlsx"

SEED = 42
REPEATS = 80
EPOCHS = 500
BATCH_SIZE = 32
LR = 0.001
PATIENCE = 5
LIBRARY = "KEGG_2021_Human"
MIN_GENES = 1

# Hyperparameter grids
L1_PENALTY_VALUES = [0.001, 0.01, 0.1]  # Pipeline will run for each value sequentially
RF_PARAMS = {'max_depth': [5, 10, 20], 'n_estimators': [200, 500]}
SVM_PARAMS = {'C': [0.1, 1, 10, 100], 'gamma': ['scale', 0.1]}

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
set_random_seed(SEED)

# ------------------------------------------------------------------------------
# Core Processing Functions
# ------------------------------------------------------------------------------

def load_data():
    """Loads and preprocesses expression and phenotype data."""
    if not EXPR_FILE.exists() or not PHENO_FILE.exists():
        raise FileNotFoundError(f"Required .xlsx files not found in: {BASE_DIR}")

    expr = pd.read_excel(EXPR_FILE, index_col=0)
    pheno = pd.read_excel(PHENO_FILE)

    s_col = next(c for c in pheno.columns if c.lower() == "sample")
    g_col = next(c for c in pheno.columns if "group" in c.lower())
    
    pheno = pheno[[s_col, g_col]].copy()
    pheno.columns = ["sample", "group"]
    pheno = pheno[pheno["group"].isin(["RA", "HE"])].copy()
    
    common_samples = [c for c in expr.columns if str(c) in pheno["sample"].astype(str).values]
    expr = expr[common_samples]
    pheno = pheno.set_index("sample").loc[common_samples].reset_index()
    
    y = LabelEncoder().fit_transform(pheno["group"]).astype(int)
    return expr, y

def gaussian_kernel(X1, X2=None, sigma=None):
    if X2 is None: X2 = X1
    D_sq = euclidean_distances(X1, X2, squared=True)
    if sigma is None:
        sigma = np.mean(np.sqrt(D_sq[D_sq > 0])) if np.any(D_sq > 0) else 1.0
    return np.exp(-D_sq / (2 * (sigma**2)))

def build_kernel_mlp(num_train_samples, num_kernels, l1_val):
    p_in = [Input(shape=(num_train_samples,), name=f"p_{i}") for i in range(num_kernels)]
    b_in = Input(shape=(1,), name="bias_in")
    shared = Dense(1, use_bias=False, name="shared_layer")
    projs = [shared(i) for i in p_in]
    merged = Concatenate()([Dense(1, use_bias=False)(b_in)] + projs)
    out = Dense(1, activation="sigmoid", use_bias=False, 
                kernel_regularizer=l1(l1_val), kernel_constraint=NonNeg(), name="final")(merged)
    model = Model(inputs=p_in + [b_in], outputs=out)
    model.compile(optimizer=Adam(LR), loss="binary_crossentropy")
    return model

def plot_performance(y_true, y_pred_proba, model_name, save_path):
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    auc_score = roc_auc_score(y_true, y_pred_proba)
    
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'AUC = {auc_score:.4f}')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'{model_name} ROC Curve')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()
    plt.savefig(save_path / f"roc_{model_name.lower()}.png")
    plt.close()

# ------------------------------------------------------------------------------
# Analysis Pipeline
# ------------------------------------------------------------------------------

def run_l1_experiment(l1_val, expr_df, y, gene_sets):
    """Executes the full training/validation/testing cycle for a specific L1 value."""
    l1_folder = OUTPUT_ROOT / f"L1_{str(l1_val).replace('.', '_')}"
    l1_folder.mkdir(exist_ok=True)
    
    print(f"\n>>> Starting Experiment: L1 Regularization = {l1_val}")
    
    class_weights = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y), y=y)))
    y_true_all, p_mlp, p_rf, p_svm = [], [], [], []
    selection_counts = pd.Series(0, index=list(gene_sets.keys()), dtype=int)
    expr_genes = list(expr_df.index)

    for r in range(REPEATS):
        tr_idx, te_idx = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=SEED+r)
        
        # Scaling
        scaler = StandardScaler().fit(expr_df.iloc[:, tr_idx].T)
        X_tr = scaler.transform(expr_df.iloc[:, tr_idx].T)
        X_te = scaler.transform(expr_df.iloc[:, te_idx].T)

        # Baseline Models (RF & SVM)
        rf = GridSearchCV(RandomForestClassifier(random_state=SEED, class_weight='balanced'), RF_PARAMS, cv=3, scoring='roc_auc').fit(X_tr, y[tr_idx])
        svm = GridSearchCV(SVC(probability=True, random_state=SEED, class_weight='balanced'), SVM_PARAMS, cv=3, scoring='roc_auc').fit(X_tr, y[tr_idx])
        
        p_rf.extend(rf.predict_proba(X_te)[:, 1])
        p_svm.extend(svm.predict_proba(X_te)[:, 1])

        # Kernel MLP Setup
        K_tr_list, K_te_list, used_p = [], [], []
        for pname, genes in gene_sets.items():
            common = list(set(genes) & set(expr_genes))
            if len(common) < MIN_GENES: continue
            
            idx = [expr_genes.index(g) for g in common]
            A, B = X_tr[:, idx], X_te[:, idx]
            sig = np.mean(np.sqrt(euclidean_distances(A, squared=True)[np.triu_indices(len(A), 1)])) or 1.0
            
            K_tr_list.append(np.exp(-euclidean_distances(A, squared=True) / (2 * sig**2)))
            K_te_list.append(np.exp(-euclidean_distances(B, A, squared=True) / (2 * sig**2)))
            used_p.append(pname)
        
        K_tr_stack = np.transpose(np.stack(K_tr_list), (1, 0, 2))
        K_te_stack = np.transpose(np.stack(K_te_list), (1, 0, 2))

        # Model Training
        clear_session()
        mlp = build_kernel_mlp(K_tr_stack.shape[2], K_tr_stack.shape[1], l1_val)
        
        f_tr = {f"p_{i}": K_tr_stack[:, i, :] for i in range(K_tr_stack.shape[1])} | {"bias_in": np.ones((len(X_tr), 1))}
        f_te = {f"p_{i}": K_te_stack[:, i, :] for i in range(K_te_stack.shape[1])} | {"bias_in": np.ones((len(X_te), 1))}
        
        mlp.fit(f_tr, y[tr_idx], epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, class_weight=class_weights)
        p_mlp.extend(mlp.predict(f_te, verbose=0).flatten())
        y_true_all.extend(y[te_idx])

        # Feature Selection Tracking
        weights = mlp.get_layer("final").get_weights()[0].flatten()[1:] # Exclude bias
        selection_counts.loc[np.array(used_p)[np.abs(weights) > 1e-9]] += 1
        
        print(f"  Iteration {r+1}/{REPEATS} completed.", end='\r')

    # Save results for this L1
    summary = pd.DataFrame({'y_true': y_true_all, 'MLP': p_mlp, 'RF': p_rf, 'SVM': p_svm})
    summary.to_excel(l1_folder / "raw_predictions.xlsx", index=False)
    
    selection_counts.sort_values(ascending=False).to_excel(l1_folder / "pathway_frequency.xlsx")
    
    for m in ['MLP', 'RF', 'SVM']:
        plot_performance(y_true_all, summary[m], m, l1_folder)
    
    print(f"\nAnalysis for L1={l1_val} finished. AUC (MLP): {roc_auc_score(y_true_all, p_mlp):.4f}")

# ------------------------------------------------------------------------------
# Entry Point
# ------------------------------------------------------------------------------

def main():
    print("Initializing Bio-Intelligence Pipeline...")
    try:
        expr_df, y = load_data()
        print(f"Data Loaded: {len(y)} samples, {len(expr_df)} genes.")
        
        print(f"Fetching biological library: {LIBRARY}...")
        gene_sets = get_library(name=LIBRARY, organism="Human")
        
        for l1_v in L1_PENALTY_VALUES:
            run_l1_experiment(l1_v, expr_df, y, gene_sets)
            
        print("\n" + "="*60)
        print("ALL TASKS COMPLETED SUCCESSFULLY")
        print(f"Check results in: {OUTPUT_ROOT}")
        print("="*60)

    except Exception as e:
        print(f"\nCRITICAL ERROR: {e}")

if __name__ == "__main__":
    main()

In [ ]:
# ==============================================================================
# BIOLOGICAL PATHWAY-BASED KERNEL-MLP: CROSS-DISEASE TRANSFER LEARNING
# ------------------------------------------------------------------------------
# Source Domain: Rheumatoid Arthritis (RA)
# Target Domains: Leukemia (AML) & Prostate Cancer
# Methodology: Shared Projection (w) Adaptation with Frozen Contributions (eta)
# ==============================================================================

import os
import sys
import random
import warnings
import json
import re
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.constraints import NonNeg
from tensorflow.keras.regularizers import l1
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, f1_score
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.utils.class_weight import compute_class_weight

from gseapy import get_library

# --- 0) GLOBAL CONFIGURATION (Original Parameters Preserved) ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
set_random_seed(SEED)

# Environment setup
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
warnings.filterwarnings("ignore")

# Local Path Management
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUTPUT_DIR = BASE_DIR / "transfer_results"
OUTPUT_DIR.mkdir(exist_ok=True)

# Dataset Registry (Ensure these files are in the BASE_DIR)
DATA_FILES = {
    "RA_EXPR": "combat_corrected_by_gse.xlsx",
    "RA_PHENO": "pheno_raw.xlsx",
    "LEUK_EXPR": "GSE15061_WithRa.xlsx",
    "LEUK_PHENO": "GSE15061_phenotype.xlsx",
    "PROS_EXPR": "GSE62872_EXPR.xlsx",
    "PROS_PHENO": "GSE62872_PHENO.xlsx",
    "TOP_PATHWAYS": "yolak_secim_frekansi.xlsx"
}

# Hyperparameters (As requested)
LIBRARY = "KEGG_2021_Human"
TOP_N = 320         # Analyzed top 320 pathways from selection frequency
REPEATS = 80        # Robustness through 80 independent iterations
EPOCHS_RA = 400     # Source training epochs
EPOCHS_ADAPT = 50   # Target adaptation epochs
BATCH_SIZE = 32
LR_RA = 0.001       # Learning rate for source
LR_ADAPT = 1e-4     # Specialized small learning rate for adaptation
L1_VAL = 0.001      # Sparsity regularization
MIN_GENES = 1

# ==============================================================================
# 1) DATA UTILITIES
# ==============================================================================

def pick_col(df, candidates):
    cols = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols: return cols[cand.lower()]
    return None

def fix_gene_dates(index):
    """Fixes Excel's auto-conversion of genes to dates (e.g., 1-Dec -> DEC1)."""
    pat = re.compile(r"^(\d{1,2})-([A-Za-z]{3})$")
    months = {"Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"}
    fixed = []
    for g in index.astype(str):
        m = pat.match(g)
        if m:
            day, mon = m.group(1), m.group(2).title()
            fixed.append(f"{mon.upper()}{day}" if mon in months else g)
        else: fixed.append(g)
    return pd.Index(fixed)

def read_bio_data(expr_path, pheno_path, disease_type):
    """Robust biological data loader with automated column detection."""
    df_expr = pd.read_excel(BASE_DIR / expr_path, engine="openpyxl")
    gene_col = pick_col(df_expr, ["Gen_name", "gene", "symbol", "Gene"])
    if gene_col: df_expr = df_expr.set_index(gene_col)
    else: df_expr = pd.read_excel(BASE_DIR / expr_path, index_col=0)
    
    df_expr.index = fix_gene_dates(df_expr.index.astype(str).str.strip())
    df_expr = df_expr.groupby(df_expr.index).mean() # Collapse duplicates

    df_ph = pd.read_excel(BASE_DIR / pheno_path, engine="openpyxl")
    s_col = pick_col(df_ph, ["sample", "sample_id", "Sample"])
    l_col = pick_col(df_ph, ["group", "group_raw", "label"])
    
    df_ph = df_ph[[s_col, l_col]].copy()
    df_ph.columns = ["sample", "label"]
    
    # Standardize Labels
    if disease_type == "RA": targets = ["RA", "HE"]
    elif disease_type == "LEUK": 
        df_ph["label"] = df_ph["label"].apply(lambda x: "AML" if "AML" in str(x).upper() else "HE")
        targets = ["AML", "HE"]
    elif disease_type == "PROS":
        df_ph["label"] = df_ph["label"].apply(lambda x: "PRO" if str(x).upper().startswith("PRO") else "HE")
        targets = ["PRO", "HE"]
        
    df_ph = df_ph[df_ph["label"].isin(targets)]
    common = [s for s in df_expr.columns if str(s) in df_ph["sample"].astype(str).values]
    
    X = df_expr[common]
    y = LabelEncoder().fit_transform(df_ph.set_index("sample").loc[common]["label"])
    return X, y

# ==============================================================================
# 2) ARCHITECTURE & KERNELS
# ==============================================================================

def gaussian_kernel(X1, X2=None, sigma=1.0):
    if X2 is None: X2 = X1
    D_sq = euclidean_distances(X1, X2, squared=True)
    return np.exp(-D_sq / (2 * sigma**2))

def compute_sigma(X):
    D_sq = euclidean_distances(X, X, squared=True)
    tri = D_sq[np.triu_indices(X.shape[0], k=1)]
    s = np.mean(np.sqrt(tri[tri > 0])) if np.any(tri > 0) else 1.0
    return s if np.isfinite(s) and s > 0 else 1.0

def build_model(n_support, n_paths):
    p_in = [Input(shape=(n_support,), name=f"p_{i}") for i in range(n_paths)]
    b_in = Input(shape=(1,), name="bias_in")
    
    # Layer w: Shared Projection (Pathway-internal logic)
    shared = Dense(1, use_bias=False, name="shared_projection")
    projs = [shared(i) for i in p_in]
    
    # Layer eta: Global contribution weights
    merged = Concatenate()([Dense(1, use_bias=False, name="bias_weight")(b_in)] + projs)
    out = Dense(1, activation="sigmoid", use_bias=False, 
                kernel_regularizer=l1(L1_VAL), kernel_constraint=NonNeg(), name="final_output")(merged)
    
    model = Model(inputs=p_in + [b_in], outputs=out)
    model.compile(optimizer=Adam(LR_RA), loss="binary_crossentropy")
    return model

# ==============================================================================
# 3) CORE EXECUTION PIPELINE
# ==============================================================================

def main():
    print("[RUN] Loading Multi-Disease Data Matrices...")
    ra_x, ra_y = read_bio_data(DATA_FILES["RA_EXPR"], DATA_FILES["RA_PHENO"], "RA")
    lk_x, lk_y = read_bio_data(DATA_FILES["LEUK_EXPR"], DATA_FILES["LEUK_PHENO"], "LEUK")
    pr_x, pr_y = read_bio_data(DATA_FILES["PROS_EXPR"], DATA_FILES["PROS_PHENO"], "PROS")
    
    # Universal gene alignment
    common_genes = sorted(list(set(ra_x.index) & set(lk_x.index) & set(pr_x.index)))
    print(f"[ALIGN] Unified Gene Set Size: {len(common_genes)}")
    
    # Load Pathways
    kegg = get_library(name=LIBRARY, organism="Human")
    df_freq = pd.read_excel(BASE_DIR / DATA_FILES["TOP_PATHWAYS"])
    top_p_names = df_freq.sort_values(by=pick_col(df_freq, ["Selection_Count", "Count"]), 
                                     ascending=False).head(TOP_N)[pick_col(df_freq, ["Pathway"])].tolist()

    active_pathways = []
    for p in top_p_names:
        if p in kegg:
            m = list(set(kegg[p]) & set(common_genes))
            if len(m) >= MIN_GENES: active_pathways.append((p, m))

    # Performance Tracking
    metrics = {"LEUK": [], "PROS": []}

    for r in range(REPEATS):
        tr_idx, _ = train_test_split(np.arange(ra_x.shape[1]), test_size=0.2, stratify=ra_y, random_state=SEED+r)
        
        # Kernel Construction (RA-Support Based)
        K_ra, K_lk, K_pr = [], [], []
        for _, genes in active_pathways:
            indices = [common_genes.index(g) for g in genes]
            A = StandardScaler().fit_transform(ra_x.iloc[indices, tr_idx].T)
            B = StandardScaler().fit_transform(lk_x.iloc[indices, :].T)
            C = StandardScaler().fit_transform(pr_x.iloc[indices, :].T)
            
            sig = compute_sigma(A)
            K_ra.append(gaussian_kernel(A, sigma=sig))
            K_lk.append(gaussian_kernel(B, A, sigma=sig))
            K_pr.append(gaussian_kernel(C, A, sigma=sig))

        X_ra, X_lk, X_pr = [np.transpose(np.stack(k), (1,0,2)) for k in [K_ra, K_lk, K_pr]]

        # --- STEP A: RA Source Training ---
        clear_session()
        cw = dict(enumerate(compute_class_weight("balanced", classes=[0,1], y=ra_y[tr_idx])))
        model = build_model(X_ra.shape[2], X_ra.shape[1])
        model.fit([X_ra[:, i, :] for i in range(X_ra.shape[1])] + [np.ones((len(tr_idx), 1))], 
                  ra_y[tr_idx], epochs=EPOCHS_RA, batch_size=BATCH_SIZE, verbose=0, class_weight=cw)

        # --- STEP B: Target Adaptation (Freeze eta, Update w) ---
        for target, data_x, data_y in [("LEUK", X_lk, lk_y), ("PROS", X_pr, pr_y)]:
            # Clone and Freeze
            t_model = build_model(X_ra.shape[2], X_ra.shape[1])
            t_model.set_weights(model.get_weights())
            for layer in t_model.layers:
                layer.trainable = (layer.name == "shared_projection")
            
            t_model.compile(optimizer=Adam(LR_ADAPT), loss="binary_crossentropy")
            t_model.fit([data_x[:, i, :] for i in range(data_x.shape[1])] + [np.ones((data_x.shape[0], 1))], 
                        data_y, epochs=EPOCHS_ADAPT, batch_size=BATCH_SIZE, verbose=0)
            
            preds = t_model.predict([data_x[:, i, :] for i in range(data_x.shape[1])] + [np.ones((data_x.shape[0], 1))], verbose=0)
            metrics[target].append(roc_auc_score(data_y, preds.flatten()))

        print(f"Repeat {r+1}/{REPEATS} | AML AUC: {metrics['LEUK'][-1]:.4f} | PROS AUC: {metrics['PROS'][-1]:.4f}", end='\r')
        gc.collect()

    # Final Export
    for k, v in metrics.items():
        pd.DataFrame({"AUC": v}).to_excel(OUTPUT_DIR / f"{k}_Transfer_Metrics.xlsx")
        print(f"\nFinal {k} Transfer AUC: {np.mean(v):.4f} (+/- {np.std(v):.4f})")

if __name__ == "__main__":
    main()